# Clinical RAG System — Sickle Cell Disease

A citation-aware Retrieval-Augmented Generation demo built from a single, well-documented modern source:

**Evidence-Based Management of Sickle Cell Disease: Expert Panel Report, 2014**
National Heart, Lung, and Blood Institute (NHLBI), National Institutes of Health (NIH)
Full report (161 pages): https://www.nhlbi.nih.gov/sites/default/files/publications/56-364NFULL.pdf

**Safety:** Educational use only. The assistant must not diagnose, prescribe, or replace a clinician.

**Why this source:** unlike the earlier 1970s scanned WHO bulletins, this is a modern, born-digital PDF — text extraction comes out clean, with no garbled spacing or scrambled tables.

## 1. Setup

In [ ]:
!pip install -q langchain langchain-community langchain-chroma langchain-text-splitters chromadb fastembed groq pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.6/116.6 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 382.9/382.9 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.2/19.2 MB 88.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137

## 2. Upload the PDF

Download the report first from the link above, then upload it here.

In [ ]:
from google.colab import files
print("Upload the Sickle Cell Disease PDF (56-364NFULL.pdf)")
uploaded = files.upload()
pdf_path = next(iter(uploaded))
print(f"Uploaded: {pdf_path}")

Upload the Sickle Cell Disease PDF (56-364NFULL.pdf)


Saving 56-364NFULL.pdf to 56-364NFULL.pdf
Uploaded: 56-364NFULL.pdf


## 3. Load pages and attach citation metadata

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

DOC_ID = 'nhlbi-scd-2014'
DOC_TITLE = 'Evidence-Based Management of Sickle Cell Disease: Expert Panel Report, 2014'
DOC_CITATION = 'National Heart, Lung, and Blood Institute (2014). Evidence-Based Management of Sickle Cell Disease: Expert Panel Report, 2014.'

loader = PyPDFLoader(pdf_path)
pages = loader.load()
for page in pages:
    page.metadata.update({
        'document_id': DOC_ID,
        'title': DOC_TITLE,
        'citation': DOC_CITATION,
        'page_number': page.metadata.get('page', 0) + 1,
    })
print(f'Loaded {len(pages)} pages')
print(pages[0].page_content[:300])

/tmp/ipykernel_2398/3143376751.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 161 pages



In [ ]:
import re

def looks_like_toc_or_refs(text):
    dot_leader_lines = len(re.findall(r'\.{4,}\s*\d+', text))
    return dot_leader_lines >= 2

pages = [p for p in pages if not looks_like_toc_or_refs(p.page_content)]
print(f'Pages remaining after filtering TOC/reference pages: {len(pages)}')

Pages remaining after filtering TOC/reference pages: 156


## 4. Chunk

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=['\n\n', '\n', '. ', ' ', '']
)
chunks = splitter.split_documents(pages)

counters = {}
for c in chunks:
    counters[DOC_ID] = counters.get(DOC_ID, 0) + 1
    c.metadata['chunk_id'] = f"{DOC_ID}-CH-{counters[DOC_ID]:04d}"

print(f'Created {len(chunks)} chunks from {len(pages)} pages')
print(chunks[0].metadata)

Created 684 chunks from 156 pages
{'producer': 'Adobe PDF Library 11.0', 'creator': 'Acrobat PDFMaker 11 for Word', 'creationdate': '2014-09-04T13:40:40-04:00', 'author': 'National Heart', 'category': 'Public Domain', 'company': '', 'keywords': '"National Heart, Lung, and Blood Institute; NHLBI; National Institutes of Health; NIH; U.S. Department of Health and Human Services; HHS; sickle cell disease; SCD; evidence base; expert panel; recommendations; consensus; methodology; health maintenance; in"', 'moddate': '2023-06-22T14:21:48-04:00', 'sourcemodified': 'D:20140904161530', 'subject': 'Evidence-Based Management of Sickle Cell Disease: Expert Panel, 2014', 'title': 'Evidence-Based Management of Sickle Cell Disease: Expert Panel Report, 2014', 'source': '56-364NFULL.pdf', 'total_pages': 161, 'page': 2, 'page_label': '3', 'document_id': 'nhlbi-scd-2014', 'citation': 'National Heart, Lung, and Blood Institute (2014). Evidence-Based Management of Sickle Cell Disease: Expert Panel Report,

Spot-check a few chunks before moving on:

In [ ]:
for c in chunks[:3]:
    print(c.metadata['chunk_id'], '| page', c.metadata['page_number'])
    print(c.page_content[:200].replace('\n', ' '))
    print()

nhlbi-scd-2014-CH-0001 | page 3
Evidence-Based  Management of  Sickle Cell Disease  Expert Panel Report, 2014

nhlbi-scd-2014-CH-0002 | page 10
Exhibit B–2.  PICOS Approach for Health Maintenance Chapter: Screening ...................... B–109  Exhibit B–3a.  PICOS Approach for Health Maintenance Chapter: Blood Pressure  (Question 1).........

nhlbi-scd-2014-CH-0003 | page 10
Exhibit B–5.  PICOS Approach for Hydroxyurea Chapter.................................................... B–111  Exhibit B–6. PICOS Approach for Transfusion Chapter ....................................



## 5. Clear any old Chroma collection

Prevents duplicate chunks if you re-run this notebook in the same session.

In [ ]:
import chromadb
_client = chromadb.Client()
try:
    _client.delete_collection('scd_clinical_kb')
    print('Old collection cleared')
except Exception:
    print('No existing collection to clear')

No existing collection to clear


## 6. Embeddings and vector database

In [ ]:
from langchain_chroma import Chroma
from langchain_community.embeddings.fastembed import FastEmbedEmbeddings

embedding_model = FastEmbedEmbeddings(model_name='BAAI/bge-small-en-v1.5')
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name='scd_clinical_kb',
    collection_metadata={'hnsw:space': 'cosine'}
)

TOP_K = 4
def retrieve_with_similarity(question, k=TOP_K):
    return vectorstore.similarity_search_with_relevance_scores(question, k=k)

print('Vector index ready.')

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

Vector index ready.


## 7. Retrieval test — no LLM call needed

Confirms the index retrieves relevant passages before spending any LLM budget.

In [ ]:
test_question = 'When should hydroxyurea therapy be started in adults with sickle cell anemia?'
retrieved = retrieve_with_similarity(test_question)
for rank, (d, score) in enumerate(retrieved, start=1):
    print(f"Rank {rank} | {d.metadata['chunk_id']} | page {d.metadata['page_number']} | similarity: {score:.4f}")
    print(d.page_content[:300], '\n')

Rank 1 | nhlbi-scd-2014-CH-0421 | page 95 | similarity: 0.8635
2. In adults with SCA who have three or more sickle cell-associated moderate to severe pain crises in a 12-month 
period, treat with hydroxyurea. 
(Strong Recommendation, High-Quality Evidence) 
3. In adults with SCA who have sickle cell-associated pain that interferes with daily activities and qual 

Rank 2 | nhlbi-scd-2014-CH-0385 | page 89 | similarity: 0.8425
Chapter 5: Hydroxyurea Therapy
in the Management of
Sickle Cell Disease 
Introduction 
This chapter addresses the use of hydroxyurea (also called hydroxycarbamide) in adults and children who have 
SCD.  Hydroxyurea can reduce the frequency of sickle cell-related pain and the incidence of acute chest 

Rank 3 | nhlbi-scd-2014-CH-0665 | page 156 | similarity: 0.8391
379. National Heart, Lung, and Blood Institute. Hydroxyurea to prevent organ damage in children 
with sickle cell anemia. In: ClinicalTrials.gov [Internet]. Bethesda (MD): National Library of 
Medicine (U

## 8. Secure LLM connection (Groq)

In [ ]:
import os
from getpass import getpass
from groq import Groq

if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass('Enter your Groq API key: ')

client = Groq(api_key=os.environ['GROQ_API_KEY'])
LLM_MODEL = 'openai/gpt-oss-120b'

Enter your Groq API key: ··········


## 9. Clinical prompt and citation formatter

In [ ]:
SYSTEM_PROMPT = '''You are a clinical education assistant specializing in sickle cell disease.
Use ONLY the supplied context to answer. If the context is insufficient, say exactly:
"The provided sources do not contain enough information to answer that."
Do not diagnose, prescribe, recommend drug doses, or select personalized treatment for any individual.
For concerning symptoms, advise assessment by a qualified clinician.
Every factual statement must end with a citation in this format: [Source: <citation>, p. <page_number>].'''

def format_docs(scored_docs):
    parts = []
    for doc, score in scored_docs:
        parts.append(f"({doc.metadata['citation']}, p. {doc.metadata['page_number']})\n{doc.page_content}")
    return '\n\n---\n\n'.join(parts)

## 10. Ask questions with retrieved evidence

In [ ]:
def ask_clinical_rag(question: str, k: int = TOP_K):
    scored_docs = retrieve_with_similarity(question, k=k)
    context = format_docs(scored_docs)

    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0.1,
        max_tokens=700,
        messages=[
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': f'Context:\n\n{context}\n\nQuestion: {question}'},
        ],
    )
    answer_text = response.choices[0].message.content

    return {
        'answer': answer_text,
        'retrieved_sources': [
            {
                'chunk_id': doc.metadata['chunk_id'],
                'page': doc.metadata['page_number'],
                'similarity': round(score, 4),
            }
            for doc, score in scored_docs
        ],
    }

In [ ]:
result = ask_clinical_rag('When should hydroxyurea therapy be started in adults with sickle cell anemia?')
print(result['answer'])
print('\nRetrieved sources:')
for source in result['retrieved_sources']:
    print(source)

Hydroxyurea should be initiated in adults with sickle cell anemia (SCA) when any of the following clinical situations are present:

* **Frequent pain crises** – three or more moderate‑to‑severe sickle‑cell‑associated pain episodes in the prior 12 months【Source: National Heart, Lung, and Blood Institute (2014), p. 95】.  
* **Pain that interferes with daily activities or quality of life**【Source: National Heart, Lung, and Blood Institute (2014), p. 95】.  
* **A history of severe and/or recurrent acute chest syndrome (ACS)**【Source: National Heart, Lung, and Blood Institute (2014), p. 95】.  
* **Severe symptomatic chronic anemia that limits daily activities or quality of life**【Source: National Heart, Lung, and Blood Institute (2014), p. 95】.

These indications are supported by strong recommendations (high‑ or moderate‑quality evidence) and by trial data showing that hydroxyurea reduces the frequency of painful episodes, ACS events, transfusion needs, and hospitalizations in adults with S

# Day 2 — Retrieval Optimization

Day 1 built the RAG pipeline for the NHLBI Sickle Cell Disease report. Day 2 keeps that work
and improves only the **retrieval layer**.

> **Rule for today:** do not optimize the prompt before verifying the evidence.

We will:
1. Confirm the Day 1 handoff (retrieval + metadata still work).
2. Add a lightweight section tag for display purposes.
3. Tune `Top-K` (3 / 5 / 10).
4. Compare chunk size / overlap configurations.
5. Build a 15–20 question evaluation set (5 categories).
6. Manually label retrieved chunks as relevant / not relevant.
7. Calculate Precision@3 and Precision@5.
8. Log real failure cases using the failure-mode table.
9. (Optional) Try keyword / hybrid search on one question.
10. Choose and justify a final retrieval configuration.

> Run all Day 1 cells first. Day 2 reuses the existing `pages`, `chunks`, `embedding_model`,
> `vectorstore`, and `retrieve_with_similarity` from Day 1 — it does not rebuild them.

## 11. Day 1 handoff check

Confirm retrieval returns chunk text plus citation metadata **before** tuning anything. This checks evidence selection, not answer writing — no LLM call here.

In [ ]:
handoff_question = 'When should hydroxyurea therapy be started in adults with sickle cell anemia?'

for rank, (doc, score) in enumerate(retrieve_with_similarity(handoff_question, k=5), start=1):
    print(f"Rank {rank} | Score {score:.4f} | Page {doc.metadata['page_number']} | {doc.metadata['chunk_id']}")
    print(doc.page_content[:350].replace('\n', ' '))
    print('-' * 100)

Rank 1 | Score 0.8635 | Page 95 | nhlbi-scd-2014-CH-0421
2. In adults with SCA who have three or more sickle cell-associated moderate to severe pain crises in a 12-month  period, treat with hydroxyurea.  (Strong Recommendation, High-Quality Evidence)  3. In adults with SCA who have sickle cell-associated pain that interferes with daily activities and quality of life, treat with  hydroxyurea.  (Strong Rec
----------------------------------------------------------------------------------------------------
Rank 2 | Score 0.8425 | Page 89 | nhlbi-scd-2014-CH-0385
Chapter 5: Hydroxyurea Therapy in the Management of Sickle Cell Disease  Introduction  This chapter addresses the use of hydroxyurea (also called hydroxycarbamide) in adults and children who have  SCD.  Hydroxyurea can reduce the frequency of sickle cell-related pain and the incidence of acute chest  syndrome (ACS).  A brief overview of these compl
-----------------------------------------------------------------------------------

## 12. Section tagging (for display only)

Day 1's metadata has `document_id`, `page_number`, and `chunk_id`, but no `section`. Rather than
re-chunk and rebuild the vector index (which would throw away Day 1's work), we add a lightweight
keyword-based section guesser used **only for display** in the evidence panels below.

This is itself a real gap worth naming later under the "Metadata problems" failure mode — the
proper fix is section-aware chunking in a future pass, not this heuristic.

In [ ]:
import re

# Coarse topic buckets for the NHLBI 2014 SCD report. Order matters: first match wins.
SECTION_KEYWORDS = [
    ('Hydroxyurea Therapy',        [r'hydroxyurea']),
    ('Stroke Prevention / TCD',    [r'transcranial doppler', r'\btcd\b', r'stroke']),
    ('Acute Chest Syndrome',       [r'acute chest syndrome']),
    ('Pain Management',            [r'vaso-?occlusive', r'pain crisis', r'analgesi']),
    ('Transfusion Therapy',        [r'transfusion', r'alloimmuni']),
    ('Priapism',                   [r'priapism']),
    ('Infection Prevention',       [r'pneumococcal', r'penicillin prophylaxis', r'immuniz']),
    ('Pregnancy / Reproductive',   [r'pregnan', r'contracept']),
    ('Renal Complications',        [r'nephropathy', r'proteinuria', r'\begfr\b']),
    ('Ophthalmologic Screening',   [r'retinopathy', r'ophthalmolog']),
    ('Pulmonary Hypertension',     [r'pulmonary hypertension']),
    ('Mental Health / Screening',  [r'depression', r'anxiety', r'neurocognitive']),
]

def guess_section(text: str) -> str:
    lowered = text.lower()
    for label, patterns in SECTION_KEYWORDS:
        if any(re.search(p, lowered) for p in patterns):
            return label
    return 'General / Unclassified'

print('Section tagger ready.')

Section tagger ready.


## 13. Evidence panel

Shows exactly what the LLM would see, with full traceability: score, document, page, guessed section, and chunk ID.

In [ ]:
def show_evidence_panel(question: str, k: int = 5):
    print('CLINICAL QUERY:', question)
    print('=' * 110)
    for rank, (doc, score) in enumerate(retrieve_with_similarity(question, k=k), start=1):
        meta = doc.metadata
        print(
            f"Chunk {rank} | Score {score:.4f} | Doc {meta['document_id']} | "
            f"Page {meta['page_number']} | Section: {guess_section(doc.page_content)} | {meta['chunk_id']}"
        )
        print(doc.page_content[:450].replace('\n', ' '))
        print('-' * 110)

show_evidence_panel('When should hydroxyurea therapy be started in adults with sickle cell anemia?', k=5)

CLINICAL QUERY: When should hydroxyurea therapy be started in adults with sickle cell anemia?
Chunk 1 | Score 0.8635 | Doc nhlbi-scd-2014 | Page 95 | Section: Hydroxyurea Therapy | nhlbi-scd-2014-CH-0421
2. In adults with SCA who have three or more sickle cell-associated moderate to severe pain crises in a 12-month  period, treat with hydroxyurea.  (Strong Recommendation, High-Quality Evidence)  3. In adults with SCA who have sickle cell-associated pain that interferes with daily activities and quality of life, treat with  hydroxyurea.  (Strong Recommendation, Moderate-Quality Evidence)  4. In adults with SCA who have a history of severe and/or r
--------------------------------------------------------------------------------------------------------------
Chunk 2 | Score 0.8425 | Doc nhlbi-scd-2014 | Page 89 | Section: Hydroxyurea Therapy | nhlbi-scd-2014-CH-0385
Chapter 5: Hydroxyurea Therapy in the Management of Sickle Cell Disease  Introduction  This chapter addresses the use of hyd

## 14. Tune Top-K

- Small `k`: focused, may miss useful evidence.
- Large `k`: better coverage, but risks noise and duplicate chunks.

Compare `k = 3, 5, 10` on at least three questions. Read the chunks — don't decide from the score alone.

In [ ]:
def compare_top_k(question, k_values=(3, 5, 10)):
    for k in k_values:
        print(f'\n========== TOP-K = {k}  |  {question} ==========')
        for rank, (doc, score) in enumerate(retrieve_with_similarity(question, k=k), start=1):
            print(
                f"{rank}. score={score:.4f} | page={doc.metadata['page_number']} "
                f"| section={guess_section(doc.page_content)} | chunk={doc.metadata['chunk_id']}"
            )
            print(doc.page_content[:220].replace('\n', ' '))

topk_test_questions = [
    'When should hydroxyurea therapy be started in adults with sickle cell anemia?',
    'How is stroke risk screened in children with sickle cell disease?',
    'What vaccinations are recommended for infection prevention in sickle cell disease?',
]

for q in topk_test_questions:
    compare_top_k(q)


========== TOP-K = 3  |  When should hydroxyurea therapy be started in adults with sickle cell anemia? ==========
1. score=0.8635 | page=95 | section=Hydroxyurea Therapy | chunk=nhlbi-scd-2014-CH-0421
2. In adults with SCA who have three or more sickle cell-associated moderate to severe pain crises in a 12-month  period, treat with hydroxyurea.  (Strong Recommendation, High-Quality Evidence)  3. In adults with SCA who
2. score=0.8425 | page=89 | section=Hydroxyurea Therapy | chunk=nhlbi-scd-2014-CH-0385
Chapter 5: Hydroxyurea Therapy in the Management of Sickle Cell Disease  Introduction  This chapter addresses the use of hydroxyurea (also called hydroxycarbamide) in adults and children who have  SCD.  Hydroxyurea can r
3. score=0.8391 | page=156 | section=Hydroxyurea Therapy | chunk=nhlbi-scd-2014-CH-0665
379. National Heart, Lung, and Blood Institute. Hydroxyurea to prevent organ damage in children  with sickle cell anemia. In: ClinicalTrials.gov [Internet]. Bethesda (MD): National 

### Top-K checkpoint

Answer directly (as text, in this cell or a new one):

1. Does Top-3 contain enough evidence for these questions?
2. Does Top-10 add useful evidence, or mostly noise and repetition?
3. Which `k` would you choose for this report, and why?

1. Does Top-3 contain enough evidence?

It depends heavily on the question. For hydroxyurea (rank1=0.8635, rank2=0.8425), Top-3 is nearly sufficient — ranks 1–2 directly state the recommendation and its clinical context. But rank 3 is already a bibliography citation, not real content. Stroke screening is similar: rank 1 is a strong direct hit, but ranks 2–3 are reference-list entries — so effectively only 1 of 3 chunks carries evidence. Vaccination is the outlier and the real problem case: rank 1 is a partial/fragmentary sentence, and ranks 2–3 are bare PDF page-footer text ("112 EVIDENCE-BASED MANAGEMENT OF SICKLE CELL DISEASE..."), with zero actual clinical content. Top-3 fails outright for this question.

2. Does Top-10 add useful evidence, or mostly noise?

Mostly noise for two of three questions — the tail of Top-10 for hydroxyurea and stroke is dominated by repeated bibliography/citation chunks (author lists, journal names), which is repetition, not new evidence. But for the vaccination question, Top-10 is doing real work: the actual pneumococcal-vaccination recommendation text doesn't show up until rank 6 ("Assure that people of all ages with SCD have been vaccinated against Streptococcus pneumoniae...") and rank 7 (infant vaccination schedule). Without going past Top-5, the system would return zero usable evidence for that question.

3. Which k would you choose, and why?

k=5 as a general default — it captures the strong direct evidence for hydroxyurea and stroke without much added noise. But the vaccination case shows k alone isn't the real fix: the correct answer sits at rank 6–7 not because k was too small, but because low-value chunks (bibliography entries, bare page-footer text) are crowding out relevant content in the top ranks. Raising k to 10 papers over that problem with more tokens rather than solving it. The better fix — worth logging as a real failure case under "Correct chunk ranked too low" — is extending the TOC/reference-page filtering you already built in Day 1 to also strip bibliography and running-header/footer chunks before they ever enter the index, rather than just increasing k.

## 15. Compare chunk size and overlap

Keep the PDF, queries, embedding model, and search method fixed. Change only chunking settings.
Sizes are in **characters** (matches Day 1's `RecursiveCharacterTextSplitter`, which is
character-based unless given a token-counting length function).

In [ ]:
chunk_configs = [
    {'name': 'small', 'chunk_size': 500,  'chunk_overlap': 75},
    {'name': 'day1',  'chunk_size': 800,  'chunk_overlap': 150},
    {'name': 'large', 'chunk_size': 1100, 'chunk_overlap': 180},
]

experiment_stores = {}

for cfg in chunk_configs:
    experiment_splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg['chunk_size'],
        chunk_overlap=cfg['chunk_overlap'],
        separators=['\n\n', '\n', '. ', ' ', '']
    )
    experiment_chunks = experiment_splitter.split_documents(pages)

    counters = {}
    for c in experiment_chunks:
        counters[DOC_ID] = counters.get(DOC_ID, 0) + 1
        c.metadata['chunk_id'] = f"{DOC_ID}-{cfg['name']}-CH-{counters[DOC_ID]:04d}"

    experiment_stores[cfg['name']] = Chroma.from_documents(
        documents=experiment_chunks,
        embedding=embedding_model,
        collection_name=f"scd_experiment_{cfg['name']}",
        collection_metadata={'hnsw:space': 'cosine'}
    )
    print(f"{cfg['name']}: {len(experiment_chunks)} chunks (size={cfg['chunk_size']}, overlap={cfg['chunk_overlap']})")

small: 997 chunks (size=500, overlap=75)
day1: 684 chunks (size=800, overlap=150)
large: 491 chunks (size=1100, overlap=180)


In [ ]:
chunk_test_questions = [
    'When should hydroxyurea therapy be started in adults with sickle cell anemia?',
    'What is the recommended screening approach for stroke risk in children?',
    'How is acute chest syndrome managed?',
]

for question in chunk_test_questions:
    print(f'\n\nQUESTION: {question}')
    for config_name, store in experiment_stores.items():
        print(f'\n--- {config_name.upper()} CONFIGURATION ---')
        results = store.similarity_search_with_relevance_scores(question, k=3)
        for rank, (doc, score) in enumerate(results, start=1):
            print(
                f"{rank}. score={score:.4f} | page={doc.metadata['page_number']} | chunk={doc.metadata['chunk_id']}"
            )
            print(doc.page_content[:200].replace('\n', ' '))



QUESTION: When should hydroxyurea therapy be started in adults with sickle cell anemia?

--- SMALL CONFIGURATION ---
1. score=0.8770 | page=93 | chunk=nhlbi-scd-2014-small-CH-0604
effects of hydroxyurea in males and females  EVIDENCE-BASED MANAGEMENT OF SICKLE CELL DISEASE: EXPERT PANEL REPORT, 2014 75
2. score=0.8689 | page=95 | chunk=nhlbi-scd-2014-small-CH-0616
Hydroxyurea Treatment Recommendations  Recommendations  1. Educate all patients with SCA and their family members about hydroxyurea therapy.  (See consensus treatment  protocol on page 145).  (Consens
3. score=0.8428 | page=131 | chunk=nhlbi-scd-2014-small-CH-0796
patients with hemoglobinopathies given either cord blood or bone marrow transplantation from  an HLA-identical sibling. Blood. 2013;122(6):1072-8.  6. Lanzkron S, Strouse JJ, Wilson R, Beach MC, Haywo

--- DAY1 CONFIGURATION ---
1. score=0.8635 | page=95 | chunk=nhlbi-scd-2014-day1-CH-0421
2. In adults with SCA who have three or more sickle cell-associated moderat

### Chunking checkpoint

Choose the configuration that most consistently places complete, relevant evidence near the top.
Change only one variable at a time in any further experiment — a higher similarity score alone
does not prove clinical usefulness.

The Day1 configuration (800 char / 150 overlap) wins clearly across all three questions:

- Hydroxyurea: Day1's rank-1 chunk is the clean numbered recommendation itself. Small's rank-1 is a page-header fragment ("effects of hydroxyurea in males and females EVIDENCE-BASED...") — noise, not content — pushing the real recommendation to rank 2. Large's rank-1 discusses shared decision-making, which is adjacent but not the core threshold criterion.
- Stroke screening: Day1's rank-1 gives the explicit recommendation (annual TCD screening up to age 10, transfusion until 18) — the single most direct answer of any config tested. Small and Large both surface similar content but rank it slightly lower, with more redundant phrasing.
- Acute chest syndrome — the clearest failure case: Small's rank-1 chunk is about splenectomy, and Large's rank-1 chunk is about splenic sequestration — both wrong-topic hits with deceptively high scores. Day1 is the only config whose rank-1 result is actually the ACS background/definition section.

## 16. Build the evaluation set (15–20 questions)

Five categories, per the lab checklist: **direct**, **paraphrased**, **abbreviation/threshold**,
**diagnosis/management-process**, and **out-of-scope**. `in_scope=False` questions test whether
the system correctly refuses instead of forcing an answer from unrelated context.

Fill in `expected_page` / `expected_section` after you've located the real answer in the PDF —
that's what makes the test repeatable rather than post-hoc.

In [ ]:
evaluation_questions = [
    # --- Direct ---
    {'question': 'When should hydroxyurea therapy be started in adults with sickle cell anemia?',
     'category': 'direct', 'in_scope': True, 'expected_section': 'Hydroxyurea Therapy'},
    {'question': 'How is stroke risk screened in children with sickle cell disease?',
     'category': 'direct', 'in_scope': True, 'expected_section': 'Stroke Prevention / TCD'},
    {'question': 'How is acute chest syndrome diagnosed and managed?',
     'category': 'direct', 'in_scope': True, 'expected_section': 'Acute Chest Syndrome'},
    {'question': 'What vaccinations are recommended for infection prevention in sickle cell disease?',
     'category': 'direct', 'in_scope': True, 'expected_section': 'Infection Prevention'},
    {'question': 'What is the recommended management for priapism in sickle cell disease?',
     'category': 'direct', 'in_scope': True, 'expected_section': 'Priapism'},

    # --- Paraphrased (same intent, different wording) ---
    {'question': 'At what point should a patient with sickle cell anemia begin hydroxyurea?',
     'category': 'paraphrased', 'in_scope': True, 'expected_section': 'Hydroxyurea Therapy'},
    {'question': 'Which imaging test is used to identify children at high risk of stroke?',
     'category': 'paraphrased', 'in_scope': True, 'expected_section': 'Stroke Prevention / TCD'},
    {'question': 'What steps should clinicians take when a patient shows signs of acute chest syndrome?',
     'category': 'paraphrased', 'in_scope': True, 'expected_section': 'Acute Chest Syndrome'},
    {'question': 'What can be done to lower the risk of alloimmunization from transfusion?',
     'category': 'paraphrased', 'in_scope': True, 'expected_section': 'Transfusion Therapy'},

    # --- Abbreviation / threshold (precision tests) ---
    {'question': 'What does TCD stand for and what is it used for in sickle cell disease?',
     'category': 'abbreviation', 'in_scope': True, 'expected_section': 'Stroke Prevention / TCD'},
    {'question': 'What velocity on transcranial Doppler indicates elevated stroke risk?',
     'category': 'threshold', 'in_scope': True, 'expected_section': 'Stroke Prevention / TCD'},
    {'question': 'What hemoglobin S percentage threshold is recommended for chronic transfusion targets?',
     'category': 'threshold', 'in_scope': True, 'expected_section': 'Transfusion Therapy'},

    # --- Diagnosis / management process ---
    {'question': 'What laboratory monitoring is recommended for a patient on hydroxyurea?',
     'category': 'process', 'in_scope': True, 'expected_section': 'Hydroxyurea Therapy'},
    {'question': 'How is renal complication risk assessed in sickle cell disease patients?',
     'category': 'process', 'in_scope': True, 'expected_section': 'Renal Complications'},
    {'question': 'What contraceptive options are recommended for women with sickle cell disease?',
     'category': 'process', 'in_scope': True, 'expected_section': 'Pregnancy / Reproductive'},
    {'question': 'How often should patients be screened for retinopathy?',
     'category': 'process', 'in_scope': True, 'expected_section': 'Ophthalmologic Screening'},

    # --- Out-of-scope (trust / refusal test) ---
    {'question': 'What is the recommended first-line treatment for type 2 diabetes?',
     'category': 'out_of_scope', 'in_scope': False, 'expected_section': None},
    {'question': 'What chemotherapy regimen is used for breast cancer?',
     'category': 'out_of_scope', 'in_scope': False, 'expected_section': None},
    {'question': 'What is the recommended dosage of ibuprofen for a healthy adult with a headache?',
     'category': 'out_of_scope', 'in_scope': False, 'expected_section': None},
]

print(f'Evaluation questions: {len(evaluation_questions)}')
for i, item in enumerate(evaluation_questions, start=1):
    flag = 'IN-SCOPE ' if item['in_scope'] else 'OUT-OF-SCOPE'
    print(f"{i:2d}. [{flag}] [{item['category']}] {item['question']}")

Evaluation questions: 19
 1. [IN-SCOPE ] [direct] When should hydroxyurea therapy be started in adults with sickle cell anemia?
 2. [IN-SCOPE ] [direct] How is stroke risk screened in children with sickle cell disease?
 3. [IN-SCOPE ] [direct] How is acute chest syndrome diagnosed and managed?
 4. [IN-SCOPE ] [direct] What vaccinations are recommended for infection prevention in sickle cell disease?
 5. [IN-SCOPE ] [direct] What is the recommended management for priapism in sickle cell disease?
 6. [IN-SCOPE ] [paraphrased] At what point should a patient with sickle cell anemia begin hydroxyurea?
 7. [IN-SCOPE ] [paraphrased] Which imaging test is used to identify children at high risk of stroke?
 8. [IN-SCOPE ] [paraphrased] What steps should clinicians take when a patient shows signs of acute chest syndrome?
 9. [IN-SCOPE ] [paraphrased] What can be done to lower the risk of alloimmunization from transfusion?
10. [IN-SCOPE ] [abbreviation] What does TCD stand for and what is it used 

## 17. Manual relevance labeling

For every question, the cell retrieves Top-5 chunks. Read each chunk and type:

- `y` — contains evidence that helps answer the question.
- `n` — unrelated, too vague, or missing the needed evidence.

This is genuinely manual. The code does not infer relevance from score or page number — that
defeats the point of the exercise. Run this cell in Colab and label honestly; it will pause for
input at each chunk.

In [ ]:
import sys

manual_evaluation = []

for item in evaluation_questions:
    question = item['question']
    print('\n' + '=' * 110, flush=True)
    print(f"QUESTION [{item['category']}]:", question, flush=True)

    results = retrieve_with_similarity(question, k=5)

    if not item['in_scope']:
        print('OUT-OF-SCOPE CHECK: inspect whether retrieved chunks fail to genuinely support an answer.', flush=True)

    labels = []
    retrieved_rows = []

    for rank, (doc, score) in enumerate(results, start=1):
        print(f"\nRank {rank} | score={score:.4f} | page={doc.metadata['page_number']} "
              f"| section={guess_section(doc.page_content)} | {doc.metadata['chunk_id']}", flush=True)
        print(doc.page_content[:500].replace('\n', ' '), flush=True)

        # Prompt directly within input to ensure immediate display
        label = input('Relevant? (y/n): ').strip().lower()
        while label not in ['y', 'n']:
            label = input('Please enter "y" or "n": ').strip().lower()

        labels.append(1 if label == 'y' else 0)
        retrieved_rows.append({
            'chunk_id': doc.metadata['chunk_id'],
            'page': doc.metadata['page_number'],
            'score': round(score, 4),
        })

    manual_evaluation.append({
        'question': question,
        'category': item['category'],
        'in_scope': item['in_scope'],
        'labels': labels,
        'retrieved': retrieved_rows,
    })

print('\nLabeling complete for', len(manual_evaluation), 'questions.', flush=True)


QUESTION [direct]: When should hydroxyurea therapy be started in adults with sickle cell anemia?

Rank 1 | score=0.8635 | page=95 | section=Hydroxyurea Therapy | nhlbi-scd-2014-CH-0421
2. In adults with SCA who have three or more sickle cell-associated moderate to severe pain crises in a 12-month  period, treat with hydroxyurea.  (Strong Recommendation, High-Quality Evidence)  3. In adults with SCA who have sickle cell-associated pain that interferes with daily activities and quality of life, treat with  hydroxyurea.  (Strong Recommendation, Moderate-Quality Evidence)  4. In adults with SCA who have a history of severe and/or recurrent ACS, treat with hydroxyurea.*  (Strong Re
Relevant? (y/n): y

Rank 2 | score=0.8425 | page=89 | section=Hydroxyurea Therapy | nhlbi-scd-2014-CH-0385
Chapter 5: Hydroxyurea Therapy in the Management of Sickle Cell Disease  Introduction  This chapter addresses the use of hydroxyurea (also called hydroxycarbamide) in adults and children who have  SCD.  Hyd

## 18. Calculate Precision@3 and Precision@5

$$\text{Precision@K} = \frac{\text{relevant chunks in the first K results}}{K}$$

We average only in-scope questions. Out-of-scope questions are reported separately as a safety
check: any chunk marked relevant there is a sign the system could be fooled into treating
unrelated content as supporting evidence.

In [ ]:
def precision_at_k(labels, k):
    return sum(labels[:k]) / k

metric_rows = []
out_of_scope_hits = []

for row in manual_evaluation:
    if row['in_scope']:
        p3 = precision_at_k(row['labels'], 3)
        p5 = precision_at_k(row['labels'], 5)
        metric_rows.append({'question': row['question'], 'category': row['category'],
                             'Precision@3': p3, 'Precision@5': p5})
        print(f"P@3={p3:.2f} | P@5={p5:.2f} | [{row['category']}] {row['question']}")
    else:
        hits = sum(row['labels'])
        out_of_scope_hits.append(hits)
        print(f"OUT-OF-SCOPE | chunks incorrectly marked relevant: {hits}/5 | {row['question']}")

if metric_rows:
    average_p3 = sum(r['Precision@3'] for r in metric_rows) / len(metric_rows)
    average_p5 = sum(r['Precision@5'] for r in metric_rows) / len(metric_rows)
    print(f"\nAverage Precision@3: {average_p3:.3f}")
    print(f"Average Precision@5: {average_p5:.3f}")

if out_of_scope_hits:
    print(f"Average false-relevance rate on out-of-scope questions: "
          f"{sum(out_of_scope_hits) / (len(out_of_scope_hits) * 5):.3f}")

P@3=0.33 | P@5=0.40 | [direct] When should hydroxyurea therapy be started in adults with sickle cell anemia?
P@3=0.33 | P@5=0.40 | [direct] How is stroke risk screened in children with sickle cell disease?
P@3=0.67 | P@5=0.60 | [direct] How is acute chest syndrome diagnosed and managed?
P@3=0.00 | P@5=0.00 | [direct] What vaccinations are recommended for infection prevention in sickle cell disease?
P@3=0.00 | P@5=0.00 | [direct] What is the recommended management for priapism in sickle cell disease?
P@3=0.67 | P@5=0.60 | [paraphrased] At what point should a patient with sickle cell anemia begin hydroxyurea?
P@3=1.00 | P@5=0.60 | [paraphrased] Which imaging test is used to identify children at high risk of stroke?
P@3=0.33 | P@5=0.40 | [paraphrased] What steps should clinicians take when a patient shows signs of acute chest syndrome?
P@3=0.67 | P@5=0.40 | [paraphrased] What can be done to lower the risk of alloimmunization from transfusion?
P@3=0.33 | P@5=0.40 | [abbreviation] What does

## 19. Retrieval failure log

Name the failure before trying to fix it. Reference table from the workshop:

| Failure mode | Symptom | Fix |
|---|---|---|
| Wrong topic | Medically related, but answers a different question | Better query formulation, metadata filtering, hybrid search |
| Missing context | Chunk has part of a recommendation, not the surrounding criteria | Increase chunk size, add overlap, section-aware chunking |
| Duplicate chunks | Top-K returns near-identical evidence repeatedly | Reduce overlap, deduplicate, diversity retrieval, page/section filtering |
| Exact term missed | Semantic search under-ranks an acronym, drug, or threshold | Keyword search, hybrid retrieval |
| Correct chunk ranked too low | Right chunk exists, but sits at rank 7–8 | Reranking, better chunking, stronger embedding model |
| Irrelevant high-score chunk | High similarity, low clinical relevance | Don't trust score alone; add reranking or human review |
| Metadata problems | Correct text, but page/section missing | Fix the chunking/metadata pipeline — becomes a Day 3 citation problem |

Document at least one **real** case you actually saw above (not a hypothetical).

In [ ]:
failure_log = [
    {
        "question": "What is the recommended dosage of ibuprofen for a healthy adult with a headache?",
        "failure_mode": "Wrong topic",
        "symptom": "Retrieved SCD vaso-occlusive crisis (VOC) analgesic protocols containing NSAIDs instead of general adult headache dosing guidelines.",
        "chunk_ids_involved": ["nhlbi-scd-2014-CH-0204", "nhlbi-scd-2014-CH-0208"],
        "proposed_fix": "Better query formulation, metadata filtering, hybrid search",
    },
    {
        "question": "What chemotherapy regimen is used for breast cancer?",
        "failure_mode": "Wrong topic",
        "symptom": "Retrieved SCD transfusion regimens and general preventive screening guidelines (mammography/HPV) due to lexical overlap on 'regimen' and 'breast cancer'.",
        "chunk_ids_involved": ["nhlbi-scd-2014-CH-0674", "nhlbi-scd-2014-CH-0170"],
        "proposed_fix": "Better query formulation, metadata filtering, hybrid search",
    },
    {
        "question": "How often should patients be screened for retinopathy?",
        "failure_mode": "Irrelevant high-score chunk",
        "symptom": "Rank 4 scored high (0.7509) by matching generic pediatric preventive screenings (amblyopia, obesity, HCV) rather than SCD retinal screening intervals.",
        "chunk_ids_involved": ["nhlbi-scd-2014-CH-0165"],
        "proposed_fix": "Don't trust score alone; add reranking or section metadata filtering",
    },
]

for entry in failure_log:
    print(f"[{entry['failure_mode']}] {entry['question']}")
    print(f"  Symptom: {entry['symptom']}")
    print(f"  Chunks: {entry['chunk_ids_involved']}")
    print(f"  Fix: {entry['proposed_fix']}\n")

[Wrong topic] What is the recommended dosage of ibuprofen for a healthy adult with a headache?
  Symptom: Retrieved SCD vaso-occlusive crisis (VOC) analgesic protocols containing NSAIDs instead of general adult headache dosing guidelines.
  Chunks: ['nhlbi-scd-2014-CH-0204', 'nhlbi-scd-2014-CH-0208']
  Fix: Better query formulation, metadata filtering, hybrid search

[Wrong topic] What chemotherapy regimen is used for breast cancer?
  Symptom: Retrieved SCD transfusion regimens and general preventive screening guidelines (mammography/HPV) due to lexical overlap on 'regimen' and 'breast cancer'.
  Chunks: ['nhlbi-scd-2014-CH-0674', 'nhlbi-scd-2014-CH-0170']
  Fix: Better query formulation, metadata filtering, hybrid search

[Irrelevant high-score chunk] How often should patients be screened for retinopathy?
  Symptom: Rank 4 scored high (0.7509) by matching generic pediatric preventive screenings (amblyopia, obesity, HCV) rather than SCD retinal screening intervals.
  Chunks: ['nhlbi-sc

## 20. Optional — keyword / hybrid search on one question

Semantic search can under-rank an exact term (drug name, acronym, numeric threshold). BM25
keyword search is a cheap way to check whether that's happening on a specific question.

In [ ]:
!pip install -q rank-bm25

In [ ]:
from rank_bm25 import BM25Okapi

def build_bm25_index(chunk_list):
    tokenized = [c.page_content.lower().split() for c in chunk_list]
    return BM25Okapi(tokenized), chunk_list

bm25_index, bm25_chunks = build_bm25_index(chunks)

def keyword_search(question, k=5):
    scores = bm25_index.get_scores(question.lower().split())
    ranked = sorted(zip(bm25_chunks, scores), key=lambda x: x[1], reverse=True)[:k]
    return ranked

def hybrid_search(question, k=5, semantic_weight=0.5):
    semantic_results = {doc.metadata['chunk_id']: (doc, score)
                         for doc, score in retrieve_with_similarity(question, k=k * 2)}
    keyword_results = keyword_search(question, k=k * 2)
    max_kw = max((s for _, s in keyword_results), default=1) or 1

    combined = {}
    for doc, score in semantic_results.values():
        combined[doc.metadata['chunk_id']] = {'doc': doc, 'score': semantic_weight * score}
    for doc, score in keyword_results:
        norm_score = score / max_kw
        cid = doc.metadata['chunk_id']
        if cid in combined:
            combined[cid]['score'] += (1 - semantic_weight) * norm_score
        else:
            combined[cid] = {'doc': doc, 'score': (1 - semantic_weight) * norm_score}

    ranked = sorted(combined.values(), key=lambda x: x['score'], reverse=True)[:k]
    return [(r['doc'], r['score']) for r in ranked]

# Test on a question likely to contain an exact term (threshold / abbreviation)
compare_question = 'What velocity on transcranial Doppler indicates elevated stroke risk?'

print('--- SEMANTIC ---')
for rank, (doc, score) in enumerate(retrieve_with_similarity(compare_question, k=5), start=1):
    print(f"{rank}. score={score:.4f} | {doc.metadata['chunk_id']}")
    print(doc.page_content[:200].replace('\n', ' '))

print('\n--- KEYWORD (BM25) ---')
for rank, (doc, score) in enumerate(keyword_search(compare_question, k=5), start=1):
    print(f"{rank}. score={score:.4f} | {doc.metadata['chunk_id']}")
    print(doc.page_content[:200].replace('\n', ' '))

print('\n--- HYBRID ---')
for rank, (doc, score) in enumerate(hybrid_search(compare_question, k=5), start=1):
    print(f"{rank}. score={score:.4f} | {doc.metadata['chunk_id']}")
    print(doc.page_content[:200].replace('\n', ' '))

--- SEMANTIC ---
1. score=0.7988 | nhlbi-scd-2014-CH-0133
was considered moderate to high.   In an observational study of 274 patients, the cumulative incidence of conversion from a normal TCD velocity  (<170 cm/sec) to a conditional TCD velocity (170–199 m/
2. score=0.7983 | nhlbi-scd-2014-CH-0578
100. Silva GS, Vicari P, Figueiredo MS, Carrete H, Jr., Idagawa MH, Massaro AR. Brain magnetic  resonance imaging abnormalities in adult patients with sickle cell disease: correlation with  transcrani
3. score=0.7980 | nhlbi-scd-2014-CH-0455
Exhibit 17. Acute Complications—Consensus Recommendations When Transfusion Is Not  Indicated  Indication    Asymptomatic anemia  Acute kidney injury, unless multisystem organ failure (MSOF)  Exhibit
4. score=0.7692 | nhlbi-scd-2014-CH-0579
history of conditional transcranial Doppler flow velocities in children with sickle cell anaemia.  Br J Haematol. 2008;142(1):94-9.  103. Makani J, Kirkham FJ, Komba A, Ajala-Agbo T, Otieno G, Fegan G
5. score=0.7644

## 21. Final retrieval configuration

Based on the Top-K comparison, chunk-config comparison, and Precision@K results above, record
your chosen configuration and the reasoning. This should reference actual numbers from Sections
14, 15, and 18 — not a guess.

In [ ]:
FINAL_CONFIG = {
    'top_k': 7,          # e.g. 5 — fill in from Section 14
    'chunk_size': 800,     # e.g. 800 — fill in from Section 15
    'chunk_overlap': 150,  # e.g. 150
    'strategy': 'hybrid', # 'semantic' | 'keyword' | 'hybrid' — fill in from Section 20 if used
}

FINAL_JUSTIFICATION = """
Concerning the retrieval configuration:

1.  **Top-K (k=7):** While `k=5` was generally effective, the analysis in Section 14 showed that some relevant information, particularly for questions like vaccinations, appeared at ranks 6 or 7. Increasing `k` to 7 provides a better balance, capturing more relevant chunks without introducing excessive noise from bibliography or reference list entries. This helps ensure better coverage for a wider range of clinical questions.

2.  **Chunking (Chunk Size=800, Chunk Overlap=150):** The 'Day1' configuration (chunk size of 800 characters and overlap of 150 characters) was the clear winner in the comparison performed in Section 15. This configuration consistently placed complete and directly relevant evidence at the top ranks for critical questions (e.g., hydroxyurea, stroke screening, acute chest syndrome management). In contrast, 'small' chunks often fragmented important information, and 'large' chunks sometimes led to less precise retrieval by including too much extraneous context or misidentifying the primary topic.

3.  **Strategy (Hybrid Search):** Hybrid search is chosen to leverage the strengths of both semantic and keyword-based retrieval. As demonstrated in Section 20, for questions involving specific terms, acronyms, or numerical thresholds (like 'transcranial Doppler velocity'), keyword search can ensure that lexically precise matches are highly ranked. Combining this with semantic search helps maintain performance for more conceptual queries, offering a robust approach to diverse clinical questions.

"""

print(FINAL_CONFIG)
print(FINAL_JUSTIFICATION)

{'top_k': 7, 'chunk_size': 800, 'chunk_overlap': 150, 'strategy': 'hybrid'}

Concerning the retrieval configuration:

1.  **Top-K (k=7):** While `k=5` was generally effective, the analysis in Section 14 showed that some relevant information, particularly for questions like vaccinations, appeared at ranks 6 or 7. Increasing `k` to 7 provides a better balance, capturing more relevant chunks without introducing excessive noise from bibliography or reference list entries. This helps ensure better coverage for a wider range of clinical questions.

2.  **Chunking (Chunk Size=800, Chunk Overlap=150):** The 'Day1' configuration (chunk size of 800 characters and overlap of 150 characters) was the clear winner in the comparison performed in Section 15. This configuration consistently placed complete and directly relevant evidence at the top ranks for critical questions (e.g., hydroxyurea, stroke screening, acute chest syndrome management). In contrast, 'small' chunks often fragmented important i

### End-of-Day-2 checklist

- [+] Correct evidence frequently appears in Top-K
- [~] Best evidence is reasonably high in the ranking
- [+] Retrieved metadata (document, page, section, chunk ID) is available
- [+] Similarity scores are visible and logged
- [+] A labeled 15–20 question evaluation set exists
- [+] Precision@K is calculated for at least K=3 and K=5
- [+] At least two chunk configurations were compared
- [ ] Main failure cases are documented, not just noticed

**Day 3 preview:** Day 2 answered *did we retrieve trustworthy evidence?* Day 3 asks *can the LLM
generate an answer using only that evidence, with a citation back to it?*